# Esplorazione del progetto docgenerator

Questo notebook analizza il codice sorgente del repository **docgenerator** e risponde a tre domande:

1. Quali moduli Maven compongono il progetto e quanto è grande ciascuno?
2. Quali annotazioni JAX-RS (`@GET`, `@POST`, `@Path`, ...) e Jackson (`@JsonProperty`) compaiono nel codice?
3. Quali endpoint HTTP sono definiti nel modulo di esempio `testproject`?

Usa solo la libreria standard di Python, quindi non serve installare nulla. Avvialo dalla cartella `notebooks/` o dalla radice del repository.

In [ ]:
import re
from collections import Counter
from pathlib import Path

# Trova la radice del repository (la cartella che contiene il pom.xml principale)
ROOT = Path.cwd()
while not (ROOT / "pom.xml").exists() or ROOT.name == "notebooks":
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Radice del repository non trovata")
    ROOT = ROOT.parent
print("Radice del progetto:", ROOT)

## 1. Moduli Maven

Leggiamo i `<module>` dichiarati nel `pom.xml` principale e contiamo file Java e righe di codice per ciascuno.

In [ ]:
pom = (ROOT / "pom.xml").read_text(encoding="utf-8")
modules = re.findall(r"<module>\s*([^<]+?)\s*</module>", pom)

def java_files(path):
    return sorted(path.rglob("*.java"))

def count_lines(f):
    righe = f.read_text(encoding="utf-8").splitlines()
    codice = [r for r in righe if r.strip() and not r.strip().startswith(("//", "*", "/*"))]
    return len(righe), len(codice)

print(f"{'Modulo':<28}{'File':>6}{'Righe':>8}{'Codice':>8}")
print("-" * 50)
totale = [0, 0, 0]
for m in modules:
    files = java_files(ROOT / m)
    righe = sum(count_lines(f)[0] for f in files)
    codice = sum(count_lines(f)[1] for f in files)
    totale = [totale[0] + len(files), totale[1] + righe, totale[2] + codice]
    print(f"{m:<28}{len(files):>6}{righe:>8}{codice:>8}")
print("-" * 50)
print(f"{'Totale':<28}{totale[0]:>6}{totale[1]:>8}{totale[2]:>8}")

Un semplice grafico a barre testuale rende più evidente il peso relativo dei moduli:

In [ ]:
dimensioni = {m: sum(count_lines(f)[1] for f in java_files(ROOT / m)) for m in modules}
massimo = max(dimensioni.values()) or 1
for m, n in sorted(dimensioni.items(), key=lambda kv: -kv[1]):
    barra = "█" * round(40 * n / massimo)
    print(f"{m:<28}{barra} {n}")

## 2. Annotazioni usate nel codice

Contiamo tutte le annotazioni Java (`@Qualcosa`) presenti nei sorgenti, evidenziando quelle che il docgenerator sa interpretare.

In [ ]:
INTERESSANTI = {"GET", "POST", "PUT", "DELETE", "HEAD", "Path", "PathParam",
                "QueryParam", "Produces", "Consumes", "JsonProperty"}

conteggio = Counter()
for f in java_files(ROOT):
    testo = f.read_text(encoding="utf-8")
    conteggio.update(re.findall(r"@([A-Z][A-Za-z]+)", testo))

for nome, n in conteggio.most_common():
    segno = "  <-- letta dal docgenerator" if nome in INTERESSANTI else ""
    print(f"@{nome:<28}{n:>4}{segno}")

## 3. Endpoint definiti in `testproject`

Il modulo `testproject` contiene una classe di esempio con metodi annotati JAX-RS. Ricostruiamo una tabella degli endpoint combinando il `@Path` della classe con quello dei metodi.

In [ ]:
VERBI = ("GET", "POST", "PUT", "DELETE", "HEAD")
endpoint = []

for f in java_files(ROOT / "testproject"):
    testo = f.read_text(encoding="utf-8")
    # @Path a livello di classe (quello che precede la dichiarazione "class")
    m_classe = re.search(r'@Path\("([^"]*)"\)[^{;]*?\bclass\b', testo, re.S)
    base = m_classe.group(1) if m_classe else ""
    # Blocchi di annotazioni seguiti da un metodo pubblico
    for blocco, metodo in re.findall(r"((?:\s*@[^\n]+\n)+)\s*public\s+[\w<>\[\], ]+?\s+(\w+)\s*\(", testo):
        verbo = next((v for v in VERBI if re.search(rf"@{v}\b", blocco)), None)
        if not verbo:
            continue
        m_path = re.search(r'@Path\("([^"]*)"\)', blocco)
        percorso = "/" + "/".join(p.strip("/") for p in (base, m_path.group(1) if m_path else "") if p.strip("/"))
        endpoint.append((verbo, percorso, f"{f.stem}.{metodo}"))

if endpoint:
    print(f"{'Verbo':<8}{'Percorso':<30}Metodo Java")
    print("-" * 70)
    for verbo, percorso, metodo in endpoint:
        print(f"{verbo:<8}{percorso:<30}{metodo}")
else:
    print("Nessun endpoint trovato.")

## Conclusioni

- Il codice è suddiviso in pochi moduli: `scanner` (annotation processor), `common` (modello dati condiviso), `docgenerator-maven-plugin` (generazione del report) e `testproject` (esempio).
- Le annotazioni JAX-RS e `@JsonProperty` sono esattamente ciò che lo scanner raccoglie per produrre il report `rest.html`.
- La tabella degli endpoint qui sopra è un'anteprima, ottenuta con espressioni regolari, di quello che il plugin Maven genera in modo completo durante `mvn site`.